# AquaHealth AI — Phase 12: CV experiment runner on Colab GPU

One experiment = one model × one fold × one data arm, trained by `scripts/run_cv_experiment.py` on CUDA. This notebook is the only place experiments run; the local machine is for code and manifests.

Rules baked into the code: `final_test.csv` is read for ids only; validation is always `fold_XX_validation.csv`; WITH-GAN data comes from `data/gan/fold_XX/`; a COMPLETED experiment is never retrained; `--require-cuda` aborts without a GPU.

## 1. Runtime → GPU. Clone the repository, install the pinned dependencies

In [ ]:
!nvidia-smi
import os, pathlib, subprocess
REPO_URL = 'https://github.com/kolursamith/aquahealth.git'   # adjust if your remote differs
BRANCH = 'release/final-completion'
REPO_DIR = pathlib.Path('/content/aquahealth')
if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log -1 --oneline
!pip install -q -r requirements/experiments.txt   # torch/torchvision/opencv/matplotlib pins; nothing else

## 2. CUDA and versions (abort here if there is no GPU)

In [ ]:
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('CUDA available', torch.cuda.is_available(), '| CUDA', torch.version.cuda)
assert torch.cuda.is_available(), 'No CUDA GPU: change the runtime type to GPU before continuing'
print('GPU', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

## 3. Dataset: mount Google Drive and reference the delivered datasets

Upload the delivery folder `Dataset/` (the five datasets exactly as delivered — see `data/raw/README.md`) to `MyDrive/AquaHealth/Dataset/`. Copying it to the VM disk makes epochs much faster than reading from Drive; the copy is ephemeral and must be redone after a runtime reset. Manifests come with the repository (git).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/AquaHealth')
DATASET_DRIVE = DRIVE / 'Dataset'
DATASET_LOCAL = pathlib.Path('/content/aquahealth_data/Dataset')
assert DATASET_DRIVE.is_dir(), f'{DATASET_DRIVE} not found — upload the delivered Dataset folder'
if not DATASET_LOCAL.exists():
    DATASET_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    !rsync -a --info=progress2 "{DATASET_DRIVE}/" "{DATASET_LOCAL}/"
!python scripts/link_raw_datasets.py --source "{DATASET_LOCAL}"
!ls -la data/raw

## 4. Persistent results and GAN outputs on Drive

`results/v2` and `data/gan` are symlinked to Drive so checkpoints, histories and synthetic images survive a disconnect.

In [ ]:
for local, remote in (('results/v2', DRIVE / 'results_v2'), ('data/gan', DRIVE / 'gan')):
    remote.mkdir(parents=True, exist_ok=True)
    local = pathlib.Path(local)
    if local.exists() and not local.is_symlink():
        # keep the committed matrix: copy it to Drive once, then link
        !rsync -a --ignore-existing "{local}/" "{remote}/"
        !rm -rf "{local}"
    if not local.is_symlink():
        local.symlink_to(remote)
    print(local, '->', local.resolve())
!ls results/v2 | head

## 5. Pre-flight for the requested experiment (integrity, dataset, manifests, CUDA)

In [ ]:
MODEL = 'cnn_vit_lstm'        # efficientnet_b0 | cnn_vit_lstm | yolo_efficientnet | cnn_bilstm | resnet_attention | yolo_transformer
FOLD = 1                     # 1..10
DATA_ARM = 'without_gan'     # without_gan | with_gan
!python scripts/colab_preflight.py --fold {FOLD} --data-arm {DATA_ARM} --require-cuda

## 6. (WITH-GAN only) generate the fold's synthetic training images first

Trains the fold-specific cDCGAN on `fold_XX_train.csv` only and writes `data/gan/fold_XX/`. Skip for WITHOUT-GAN.

In [ ]:
RUN_GAN = False   # set True for DATA_ARM == 'with_gan' when data/gan/fold_XX does not exist yet
if RUN_GAN:
    !python scripts/run_gan_fold.py --fold {FOLD} --epochs 30 --device cuda
    !python scripts/colab_preflight.py --fold {FOLD} --data-arm with_gan --require-cuda

## 7. INFRASTRUCTURE SMOKE TEST — NOT A PERFORMANCE RESULT

cnn_vit_lstm · fold 01 · WITHOUT-GAN · 64 training / 32 validation images · 2 epochs (1 head + 1 full) on CUDA. Exercises loading, CLAHE, model, forward/backward, optimiser, validation, metrics, checkpoints, resume and result writing. Written under `results/v2/smoke/` (never mixed with the real matrix). Do not report its metrics as accuracy.

In [ ]:
SMOKE_ID = 'smoke_cnn_vit_lstm_fold01_without_gan'
!python scripts/run_cv_experiment.py --model cnn_vit_lstm --fold 1 --data-arm without_gan \
    --config configs/cv_v2/smoke.json --require-cuda --smoke \
    --max-train-samples 64 --max-validation-samples 32 \
    --out-root results/v2/smoke --experiment-id {SMOKE_ID}
!ls -la results/v2/smoke/{SMOKE_ID} && cat results/v2/smoke/{SMOKE_ID}/run_summary.json

## 8. Launch ONE real experiment (Phase 13 — do not run in Phase 12)

Set RUN_EXPERIMENT = True only when the Phase-13 execution plan says so. `--resume` continues an interrupted run; a COMPLETED experiment is refused.

In [ ]:
RUN_EXPERIMENT = False
RESUME = False
if RUN_EXPERIMENT:
    resume_flag = '--resume' if RESUME else ''
    !python scripts/run_cv_experiment.py --model {MODEL} --fold {FOLD} --data-arm {DATA_ARM} \
        --config configs/cv_v2/default.json --require-cuda {resume_flag} \
        2>&1 | tee -a "results/v2/experiments/{MODEL}_fold{FOLD:02d}_{DATA_ARM}.console.log"
    !python scripts/build_experiment_matrix.py --refresh
    !grep -c COMPLETED results/v2/experiment_matrix.csv || true

## 9. After a disconnect

Re-run sections 1–5 (the Drive symlinks restore `results/v2` and `data/gan`), then section 8 with `RESUME = True`. `latest.pt` holds model/optimizer/scheduler/scaler/RNG state and the history; the configuration hash must match.